# 15 — MCP read-only tools

Savoroute MCP v0.13.0 uses the **same** retrieval and recommendation service as HTTP. This notebook exercises the local in-process bridge and, if installed, the official MCP SDK v2. The fixture is fictional; no performance claim is implied.

In [ ]:
from savoroute.mcp.bridge import CatalogBridge
bridge = CatalogBridge()
metadata = await bridge.metadata()
assert metadata["catalog_synthetic"] is True
metadata

In [ ]:
items = await bridge.items(kind="recipe", limit=3)
assert len(items["items"]) == 3
items["items"][0]

In [ ]:
search = await bridge.search({"query": "spicy tofu noodles", "filters": {"kind": "recipe"}})
ranking = await bridge.rank({"query": "spicy tofu noodles", "filters": {"kind": "recipe"}})
assert ranking["mode"] == "bm25_only"
[(hit["item"]["title"], hit["score"]) for hit in search["results"]]

In [ ]:
result = await bridge.recommend({
    "query": "spicy tofu noodles",
    "preferences": {"cuisines": ["Japanese"], "excluded_item_ids": ["p002"]},
})
assert all(hit["item_id"] != "p002" for hit in result["results"])
[(hit["title"], hit["explanation"]) for hit in result["results"]]

## Optional actual MCP protocol verification

The next cell performs an in-memory MCP v2 client-to-server call if `pip install -e ".[mcp]"` has been run. No TCP port, network request, or model download is needed.

In [ ]:
import importlib.util
if importlib.util.find_spec("mcp") is not None:
    from mcp import Client
    from savoroute.mcp.server import create_server
    async with Client(create_server()) as client:
        listing = await client.list_tools()
        assert len(listing.tools) == 6
        output = await client.call_tool("catalog_metadata", {})
        assert output.structured_content["catalog_items"] == 12
        print("MCP protocol verified:", [tool.name for tool in listing.tools])
else:
    print("Optional MCP SDK not installed; bridge checks above still passed")

## Design notes

- `savoroute mcp serve` listens on **stdio**, not HTTP. The process staying quiet is expected.
- The host chooses the catalog/index at startup; MCP tool calls never accept paths or URLs.
- Model inference and downloads are not available through this server.
- RRF scores are not probabilities, and catalog facts may be incomplete.

See [docs/mcp-server.md](../docs/mcp-server.md) and [ADR 0009](../docs/adr/0009-local-mcp-adapter.md).